# 03 · 실험 실행 (팀원용)

**맨 아래 `RUNNER_NAME` 만 본인 이름으로 바꾸고, 원하는 프리셋 셀을 실행하면 된다.**
결과는 자동으로 Drive 공유 원장에 쌓인다.

---

## ⚠️ 실험 전에 반드시 읽을 것

1. **폴드2024 와 폴드2023 의 HPO 순위는 Spearman −0.806 으로 역전된다.**
   27개 조합 중 두 폴드 동시 개선은 **0개**였다.
   → `채택후보` 가 떠도 **LB 개선이 보장되지 않는다.** 후보일 뿐이다.
2. **CV 이득을 LB 기대치로 옮기지 마라.**
   실측 전이율: v8 **0.21배**, v9 **0.53배**, HPO **0.06배**.
3. **블렌드 가중치는 CV 로 고르지 마라.**
   동일 모델에서 CV **+14.2** 인 설정이 LB **−11.66** 이었다. 부호가 반대다.

## 우리가 이미 아는 것 (중복 실험 방지)

| 축 | 현재 설정 | 알려진 방향 |
|---|---|---|
| XGB | d6 / mcw1500 / n600 | **크고 느리게** 갈수록 폴드2024 개선 (821.7 → 869.7) |
| LGB | leaves31 / mcs1500 | **작게** 갈수록 개선 (796.3 → 830.8 @ leaves15/mcs6000) |
| CatBoost | d6 / n2000 | **깊이면 붕괴** (d6 745 → d8 640 → d10 437). d6 가 최적 |
| cmp 성분모델 | d6 / mcw1500 / n600 | **유일하게 재튜닝 안 된 축** |

이미 기각된 것: 성분×상황 피처(−11.4), 혼합분해(−158.2), 인수분해(−29), 구종 스태킹(+1.8),
콜드스타트(−1.5), 2024 가중 강화(−94), dart(−2.1), exact(−10.8), linear_tree(실패).

In [ ]:
import os, sys
DRIVE_ROOT = '/content/drive/MyDrive/lga'          # 팀 공유 폴더
from google.colab import drive; drive.mount('/content/drive', force_remount=False)
os.environ['LGA_ROOT']   = '/content/lga'
os.environ['LGA_DATA']   = f'{DRIVE_ROOT}/data/'
os.environ['LGA_CACHE']  = '/content/lga/cache/'
os.environ['LGA_LEDGER'] = f'{DRIVE_ROOT}/ledger/'
os.environ['LGA_ASSETS'] = '/content/lga-repo/assets/'
os.environ['LGA_DEV']    = 'cuda:0'
sys.path.insert(0, '/content/lga-repo/src')

# Drive 캐시 -> 로컬 (로컬이 훨씬 빠르다)
import shutil
os.makedirs('/content/lga/cache', exist_ok=True)
for f in ['X98.parquet','tm5.parquet','oof_comp.parquet','features.parquet','aligned.parquet']:
    s, d = f'{DRIVE_ROOT}/cache/{f}', f'/content/lga/cache/{f}'
    if os.path.exists(s) and not os.path.exists(d):
        shutil.copy(s, d); print('복사', f)

import config as C, lib_lga as L, experiment as E
print(C.describe())

In [ ]:
RUNNER_NAME = '여기에_본인이름'          # ← 반드시 바꿔라
SEEDS = 3                              # 시드가 많을수록 노이즈 막대가 줄어든다 (시간은 비례)

base = E.get_baseline()
print(f"기준선  2024 {base['m24']:.1f} ±{base['m24_sd']:.1f}   2023 {base['m23']:.1f} ±{base['m23_sd']:.1f}")
print(f"실험 시드 {SEEDS} 기준 노이즈 막대(2se): "
      f"2024 ±{2*E._se(base['m24_sd'],SEEDS):.1f}   2023 ±{2*E._se(base['m23_sd'],SEEDS):.1f}")
print('\n이 막대보다 작은 차이는 전부 노이즈다.')

---
## 프리셋 1 · XGB 하이퍼파라미터 스윕

**GPU 필요 · 조합당 2~5분 · 아래 격자는 6조합 ≈ 20~30분**

XGB 는 크고 느리게 갈수록 폴드2024 가 좋아졌다. 다만 그렇게 고른 설정이
폴드2023 에서는 최악이었다 — 그래서 두 폴드를 같이 본다.

In [ ]:
df = E.run_experiment(
    name='xgb_depth_mcw',
    kind='hparam',
    model='xgb',
    grid={'max_depth': [6, 8, 10], 'min_child_weight': [1500, 6000]},
    seeds=SEEDS, runner=RUNNER_NAME,
    notes='XGB 깊이/규제 스윕',
)

In [ ]:
# 트리 수 x 학습률 (느리게 갈수록 좋았는지 재확인)
df = E.run_experiment(
    name='xgb_n_lr',
    kind='hparam', model='xgb',
    grid={'n_estimators': [600, 2000], 'learning_rate': [0.008, 0.005]},
    base_params=dict(E.DEFAULT_XGB, max_depth=10, min_child_weight=6000),
    seeds=SEEDS, runner=RUNNER_NAME,
    notes='d10/mcw6000 위에서 트리수·학습률',
)

---
## 프리셋 2 · LightGBM 스윕

**CPU (LightGBM 은 pip 휠에 GPU 가 없다) · 조합당 2~4분 · 6조합 ≈ 20분**

LGB 는 XGB 와 **반대로 작을수록** 좋았다 (leaves31 796.3 → leaves127 698.0).
자기 기준선(현재 leaves31/mcs1500)과 비교된다.

In [ ]:
df = E.run_experiment(
    name='lgb_leaves_mcs',
    kind='hparam', model='lgb',
    grid={'num_leaves': [15, 31, 63], 'min_child_samples': [1500, 6000]},
    seeds=SEEDS, runner=RUNNER_NAME,
    notes='LGB 는 작을수록 좋았다',
)

---
## 프리셋 3 · CatBoost 스윕

**GPU 필요 · 조합당 4~8분 · 3조합 ≈ 20분**

CatBoost 는 깊이를 키우면 붕괴했다 (d6 745 → d8 640 → d10 437).
**더 얕은 쪽(d4, d5)** 이 아직 안 재봤다.

In [ ]:
df = E.run_experiment(
    name='cb_depth',
    kind='hparam', model='cb',
    grid={'depth': [4, 5, 6]},
    seeds=1,                      # CatBoost 는 느리다. 시드1 이면 노이즈 막대가 커진다
    runner=RUNNER_NAME,
    notes='더 얕은 쪽 탐색',
)

---
## 프리셋 4 · 성분모델(cmp) 재튜닝  ★ 우선순위 높음

**GPU 필요 · 30~60분**

`cmp_reverse/middle/ball/strike` 4개는 **재튜닝에서 유일하게 빠진 축**이다.
이 4개의 출력이 트리 축 전체의 입력(120피처 중 6개)인데 아직 옛 설정(d6/mcw1500/n600)이다.
성분 라벨은 타깃보다 자체 신호가 2배 강하다 (reverse 1431 · ball 1778 vs y 809).

성분 OOF 를 새 설정으로 다시 만들고, 그걸 넣은 y 모델을 기준선과 비교한다.
**누수 방지**: 시즌 s 행의 OOF 는 반드시 `season < s` 로만 학습한다.

In [ ]:
import numpy as np, pandas as pd, xgboost as xgb, scipy.special as sp

CMP_NEW = dict(n_estimators=2000, learning_rate=0.005, max_depth=10, min_child_weight=6000,
               subsample=0.7, colsample_bytree=0.5, reg_lambda=50., reg_alpha=1.,
               tree_method='hist', eval_metric='logloss', verbosity=0)

def rebuild_comp_oof(prm):
    """성분 OOF 재생성. 시즌 s 는 <s 로만 학습 (폴드2024/2023 양쪽에서 누수 없음)."""
    b = L.load_base()
    RAW, season, isF = b['RAW'], b['season'], b['isF']
    BASE = L.build_base114(b)
    comp, _, _ = L.recover_labels(RAW)
    okl = comp.notna().all(1).values
    lgt = lambda p: sp.logit(np.clip(p, 1e-6, 1-1e-6))
    OOF = {c: np.full(len(RAW), np.nan, np.float32) for c in L.COMP}
    for s in range(2021, 2025):
        tr = (season < s) & ~(isF & (season <= 2022) & (s >= 2023)) & okl
        tg = season == s
        for c in L.COMP:
            m = xgb.XGBClassifier(**prm, random_state=0, device=C.DEV)
            m.fit(BASE[tr], comp[c].values[tr])
            OOF[c][tg] = lgt(m.predict_proba(BASE[tg])[:, 1])
        print(f'  시즌{s} 완료 (학습 {tr.sum():,})')
    return L.cmp_frame(OOF)

print('성분 OOF 재생성 중 (16회 학습, 20~40분)...')
new_oof = rebuild_comp_oof(CMP_NEW)
X_new = L.build_v7(oof=new_oof)
print('새 피처', X_new.shape)

In [ ]:
df = E.run_experiment(
    name='cmp_retune_d10mcw6000',
    kind='feature',
    features=X_new,          # 성분 OOF 만 교체된 120피처
    seeds=SEEDS, runner=RUNNER_NAME,
    notes='성분모델 d10/mcw6000/n2000 로 재튜닝한 OOF',
)

---
## 프리셋 5 · 새 피처 실험

`feature_fn` 을 고쳐서 아이디어를 넣어라. 반환 DataFrame 의 열이 기존 120피처에 붙는다.

**데이터 분석에서 나온 힌트**: 정보 함량은 투수 정체성에 몰려 있다.
`투수 ID 797 > 투수 커리어 제구율 607 >> 구종 212 > 시즌 147 > 타자 121 >> 상황변수 3~31`
→ **상황 변수 쪽은 이미 여러 번 실패했다.** 투수 표현을 개선하는 쪽이 승산이 높다.
특히 ID(797)와 커리어 제구율(607)의 **격차 190** 이 실신호인지 이산화 손실인지가 미해결이다.

In [ ]:
import numpy as np, pandas as pd

def my_feature(RAW, X98, b):
    """여기를 고쳐라. 반환 DataFrame 은 RAW 와 행 수·순서가 같아야 한다."""
    out = {}
    # 예: 투수 커리어 표본크기의 로그 (사전분포 신뢰도)
    out['f_car_logn'] = np.log1p(RAW.asof_pitcher_n.values).astype('float32')
    return pd.DataFrame(out, index=RAW.index)

df = E.run_experiment(
    name='내_아이디어_이름',
    kind='feature', feature_fn=my_feature,
    seeds=SEEDS, runner=RUNNER_NAME,
    notes='설명을 적어라',
)

---
## 오늘 결과 확인

In [ ]:
led = E.read_all_ledgers()
mine = led[led.runner == RUNNER_NAME] if len(led) else led
print(f'내 실험 {len(mine)}건 / 팀 전체 {len(led)}건')
if len(mine):
    display(mine[['name','params_json','m24','m23','delta24','delta23','verdict']].tail(20))
    print('\n판정 분포:'); print(mine.verdict.value_counts().to_string())